# PARC2026 — 74 M3 minimal simulator smoke

Notebook 73 training smoke PASS後、75a/75b本番trainingの前に評価系を確認します。forward/equal-dataのsmoke checkpointだけを使い、`libero_spatial` 10 tasks × 2 evaluation seeds × 3 models = **60 episodes**。promotion evidenceではありません。OpenVLAはJIT merge後にmerged weightsだけcleanupします。fresh Colabでもmodel runtime/LIBERO runtimeを再構築しますが、72a〜72d probeや1800秒benchmarkは再実行しません。失敗時の証拠はattempt単位でDriveへ保存します。LIBERO configは非対話で事前生成し、3モデルで同じ固定task/assets pathを使用します。Colabのinline Matplotlib backendは隔離venvへ持ち込まず、全simulator child processでheadless `Agg` + EGLを使用します。π0.5 v0.4.4互換性のため存在しないhard_reset CLIは渡さず、LIBERO既定のhard resetを使用します。LeRobot評価動画も無効化します。OpenVLAは固定LIBERO checkoutをvenv-local `.pth` で直接公開し、upstream legacy `setup.py/find_packages()` のnamespace packaging問題を回避します。LIBERO/robosuite 1.4.xと非互換な最新MuJoCoへの依存解決ドリフトを避けるため、3 evaluator runtimeすべてを `mujoco==3.3.1` に固定してからepisodeを開始します。π0.5/SmolVLAは各checkpointの`config.json`から期待visual feature名を検証し、hf-liberoの`image/image2`を必要時だけD10の`front/wrist`へrenameします。未知のcamera layoutは推測せず停止します。LeRobot upstream evaluatorが終了時に`eval_info.json`を書けるよう、その専用output directoryを評価開始前に作成します。disk枯渇による長時間実行後の失敗を避けるため、setup前とepisode開始前にlocal/Drive headroomを検査し、uv/pip/aptの再構築可能cacheだけをcleanupします。Hugging Face cache、D10、checkpoint、既存attempt証拠は削除しません。各evaluatorはtask完了ごとのsmall partial evidenceも保存します。OpenVLAはこのattemptで新規materializeしたmerged weightsだけを成功・失敗どちらでもcleanupし、既存LoRA/components/D10 statsは保持します。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_minimal_sim_smoke'
PIN = '1b983910509020dc2a01f1474140d72bf5664e8e'
ATTEMPT = '8'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('74 simulator smoke code:', got, flush=True)
print('74 simulator smoke attempt:', ATTEMPT, flush=True)
env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)
env['PARC_ROOT'] = str(ROOT)
env['PARC_DRIVE_ROOT'] = '/content/drive/MyDrive/parc2026-cache'
env['PARC_M3_EXECUTE'] = '1'
env['PARC_M3_SIM_SMOKE_ATTEMPT'] = ATTEMPT
env['LIBERO_CONFIG_PATH'] = str(ROOT / 'm3-libero-config/shared')
env['MPLBACKEND'] = 'Agg'
env['MUJOCO_GL'] = 'egl'
env['PYOPENGL_PLATFORM'] = 'egl'
DRIVE = Path(env['PARC_DRIVE_ROOT'])
setup_root = DRIVE / 'model-benchmark-v1/m3-simulator-minimal-smoke-v1/runtime-setup'
attempt_root = DRIVE / f'model-benchmark-v1/m3-simulator-minimal-smoke-v1/attempt-{ATTEMPT}'
setup_status = setup_root / 'runtime_setup_status.json'
setup_log = setup_root / 'runtime_setup.log'
summary = attempt_root / 'm3_minimal_simulator_smoke_summary.json'

def safe_tail(path, limit):
    try:
        if not path.is_file():
            print('missing', flush=True)
            return
        with path.open('rb') as fh:
            try:
                fh.seek(-limit, 2)
            except OSError:
                fh.seek(0)
            data = fh.read(limit)
        print(data.decode('utf-8', errors='replace'), flush=True)
    except Exception as exc:
        print(f'diagnostic read failed: {type(exc).__name__}: {exc}', flush=True)

try:
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_disk_headroom.py'), '--phase', 'pre-setup'
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_libero_noninteractive_config.py')
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_simulator_runtimes.py')
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_mujoco_compat.py')
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_disk_headroom.py'), '--phase', 'pre-eval'
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/run_m3_minimal_simulator_smoke.py')
    ], cwd=str(REPO), env=env, check=True)
except Exception:
    print('=== 74 FAILURE DIAGNOSTICS ===', flush=True)
    subprocess.run(['df', '-h', '/content', '/content/drive'], check=False)
    subprocess.run(['df', '-i', '/content', '/content/drive'], check=False)
    for path, limit in ((setup_status, 12000), (setup_log, 12000), (summary, 12000)):
        print(f'--- {path} ---', flush=True)
        safe_tail(path, limit)
    try:
        eval_logs = sorted(attempt_root.rglob('eval.log'), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    except Exception as exc:
        print(f'eval log discovery failed: {type(exc).__name__}: {exc}', flush=True)
        eval_logs = []
    if eval_logs:
        latest = eval_logs[-1]
        print(f'--- latest eval log: {latest} ---', flush=True)
        safe_tail(latest, 16000)
    partials = []
    try:
        partials = sorted(attempt_root.rglob('episode_records.partial.json'))
    except Exception as exc:
        print(f'partial evidence discovery failed: {type(exc).__name__}: {exc}', flush=True)
    for partial in partials[-3:]:
        print(f'--- partial evidence: {partial} ---', flush=True)
        safe_tail(partial, 8000)
    raise
print('=== 74 COMPLETE ===', flush=True)
print('Minimal simulator smoke only. Benchmark training/promotion/final evaluation has NOT started.', flush=True)
